In [ ]:
import kagglehub
import os
import numpy as np
import pandas as pd


# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)

In [ ]:
# Task 1: Write your code here:
import os
Q3_data_path = os.path.join(path, 'Q3_data.csv')
df = pd.read_csv(Q3_data_path)


In [ ]:
# Task 2: Write your code here:
df.head()


In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# Task 1: Write your code here:
# Inspect missing values
df.isnull().sum()

# Separate numerical and categorical columns
num_cols = df.select_dtypes(include=['int64', 'float64']).columns
cat_cols = df.select_dtypes(include=['object']).columns

# Fill missing values
for col in num_cols:
    df[col].fillna(df[col].median(), inplace=True)

for col in cat_cols:
    df[col].fillna(df[col].mode()[0], inplace=True)


In [ ]:
# Task 2: Write your code here:
# Check duplicates
df.duplicated().sum()


In [ ]:
# Remove duplicates
df.drop_duplicates(inplace=True)


In [ ]:
# Task 3: Write your code here:
# One-Hot Encoding
df = pd.get_dummies(df, drop_first=True)


In [ ]:
# Task 4: Write your code here:
from sklearn.preprocessing import StandardScaler

# Identify target column (binary column with values 0 and 1)
for col in df.columns:
    if set(df[col].unique()).issubset({0, 1}) and df[col].nunique() == 2:
        target_col = col
        break

print("Target column:", target_col)

# Split features and target
X = df.drop(columns=[target_col])
y = df[target_col]

# Scale features
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)


In [ ]:
# Task 5: Write your code here:
y.value_counts(normalize=True)


In [ ]:
# Task 1: Write your code here:
X = df.drop(columns=[target_col])
y = df[target_col]


In [ ]:
# Task 2,3,4,5: Write your code here:from sklearn.model_selection import StratifiedKFold
!pip install catboost

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score, accuracy_score
from catboost import CatBoostClassifier
import numpy as np

# Use StratifiedKFold for classification
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

scores = []

for train_idx, val_idx in skf.split(X, y):
    X_train, X_val = X.iloc[train_idx], X.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    # Train CatBoost model
    model = CatBoostClassifier(
        iterations=200,
        learning_rate=0.1,
        depth=6,
        verbose=0,
        random_state=42
    )

    model.fit(X_train, y_train)

    # Predictions
    y_pred = model.predict(X_val)

    # Use F1 Score (appropriate for imbalanced data)
    score = f1_score(y_val, y_pred)
    scores.append(score)

# Print averaged score
print("Average F1 Score across folds:", np.mean(scores))


In [ ]:
# Task 1: Write your code here:
from catboost import CatBoostClassifier

final_model = CatBoostClassifier(
    iterations=300,
    learning_rate=0.1,
    depth=6,
    verbose=0,
    random_state=42
)

final_model.fit(X, y)


In [ ]:
# Task 2: Write your code here:
import pandas as pd

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': final_model.get_feature_importance()
}).sort_values(by='Importance', ascending=False)


In [ ]:
# Task Bonus: Write your code here:
# Get the golden feature name (most important feature)
golden_feature_name = feature_importance.iloc[0]['Feature']

# Create feature matrix using only the golden feature
X_golden = df[[golden_feature_name]]

# Verify
print("Golden Feature:", golden_feature_name)
print("Shape of X_golden:", X_golden.shape)



In [ ]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from catboost import CatBoostClassifier
import numpy as np

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

golden_scores = []

for train_idx, val_idx in skf.split(X_golden, y):
    X_train, X_val = X_golden.iloc[train_idx], X_golden.iloc[val_idx]
    y_train, y_val = y.iloc[train_idx], y.iloc[val_idx]

    model = CatBoostClassifier(
        iterations=300,
        learning_rate=0.1,
        depth=6,
        verbose=0,
        random_state=42
    )

    model.fit(X_train, y_train)

    y_pred = model.predict(X_val)
    score = accuracy_score(y_val, y_pred)
    golden_scores.append(score)


In [ ]:
print("Average Accuracy (Golden Feature Only):", np.mean(golden_scores))
print("Average Accuracy (Full Model):", np.mean(scores))
